# 七品小时数据清洗与质量检查

版本 steamdt-cleaning-notebook-1.0。读取本地冻结数据，保留缺失与真实大幅涨跌，不联网、不训练。默认只查看已有清洗版本。

方法依据见[近期调研](../../docs/research/DATA_METHODS_20261019.md)，字段及操作见[清洗说明](../../docs/data/STEAMDT_CLEANING.md)。


In [ ]:
from pathlib import Path
from contextlib import closing
import sys, json, sqlite3, html
from IPython.display import display, Markdown, HTML
sys.dont_write_bytecode = True
PROJECT = next((p for p in [Path.cwd(), *Path.cwd().parents] if (p / "scripts/clean_steamdt_data.py").is_file()), None)
if PROJECT is None:
    raise RuntimeError("请先在VS Code打开 F:/Codex/CS_YouOweMe")
if str(PROJECT) not in sys.path:
    sys.path.insert(0, str(PROJECT))
from scripts.research.data import clean_hourly as cleaner
print("项目：", PROJECT)


## 1. 查看已有结果，或明确新建版本

RUN_CLEAN=False默认只读。更新时先在采集Notebook完成有限daily，然后将此处改为True并运行本单元格；旧清洗版本不覆盖。


In [ ]:
RUN_CLEAN = False
WORKSPACE = "research_20261016_v1"
OUTPUT_ID = None
CLEANING = cleaner.build(WORKSPACE, OUTPUT_ID) if RUN_CLEAN else cleaner.status()
print("状态：", CLEANING["status"])
if CLEANING.get("output_dir"):
    OUTPUT = Path(CLEANING["output_dir"])
    print("版本：", CLEANING["output_id"])
    print("清洗库：", OUTPUT / CLEANING["database"])
    print("报告：", OUTPUT / CLEANING["report"])
else:
    print("还没有清洗版本；需要时将RUN_CLEAN改成True。")


## 2. 七品质量报告

来源观察、网格占位和合法价格分开计数。首次观察之前的空白与内部缺口分开；NULL不是0。95%覆盖且最近24小时完整是预设质量门槛，尚未验证预测收益，不等于训练准入。


In [ ]:
if CLEANING.get("output_dir"):
    display(Markdown((OUTPUT / CLEANING["report"]).read_text(encoding="utf-8")))
    summary = CLEANING["summary"]
    display(Markdown("```json\n" + json.dumps({k:v for k,v in summary.items() if k != "items"}, ensure_ascii=False, indent=2) + "\n```"))


## 3. 检查大幅涨跌：保留价格，只做标记

这些是统计事件，不能据此认定操盘、情绪或错误。例子来自已冻结历史，不是当前报价。


In [ ]:
if CLEANING.get("output_dir"):
    database = OUTPUT / CLEANING["database"]
    with closing(sqlite3.connect(database.as_uri() + "?mode=ro", uri=True)) as connection:
        connection.row_factory = sqlite3.Row
        EVENTS = [dict(r) for r in connection.execute("SELECT market_hash_name,at_beijing,close_cny,large_up_move,large_down_move,quality_flags FROM hourly WHERE large_up_move=1 OR large_down_move=1 ORDER BY bucket_time DESC,market_hash_name LIMIT 15")]
    if EVENTS:
        fields = list(EVENTS[0])
        markup = "<table><tr>" + "".join("<th>"+html.escape(k)+"</th>" for k in fields) + "</tr>"
        markup += "".join("<tr>"+"".join("<td>"+html.escape(str(r[k]))+"</td>" for k in fields)+"</tr>" for r in EVENTS)
        display(HTML(markup+"</table>"))
    else:
        print("该清洗版未出现达到预设阈值的大涨跌。")


## 下一步

清洗已完成，下一步才建立新预测特征/标签。可比较缺失感知特征与预训练时序模型，但本页不下载权重或训练。旧cyrex_daily_samples/forecast入口仍按原严格规范读取旧来源，没有自动切换到这个清洗库，已有测试成绩保留。
